In [1]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from pprint import pprint

import src.utils as utils
import src.visuals as visual

from src.models import PINN
from src.loss import NavierStokesLoss
from src.dataloader import load_data, gen_dataloaders
from src.train import train_collocation

In [2]:
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

TRAIN_DATA_PATH = pathlib.Path("data/sampled_data/frac_1")
TRAIN_FILE_NAME = "data"

TEST_DATA_PATH = pathlib.Path("data/raw_data")
TEST_FILE_NAME = "extrapolation_test.csv"

INPUT_COL_NAMES = ["time", "re", "x", "y"]
TARGET_COL_NAMES = ["U_x", "U_y", "p"]

EPOCHS = 300
BATCH_SIZE = 256
LEARNING_RATE = 1e-3

N_COLLOCATION = 8192

PHYSICS_WEIGHTS = [
    0.0,
    0.01,
    0.05,
    0.1,
    0.5,
    1.0
]

LR_FACTOR = 0.5
LR_PATIENCE = 15

print(f"Datasets: {TRAIN_FILE_NAME}, {TEST_FILE_NAME}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Collocation points: {N_COLLOCATION}")
print(f"Physics weights: {PHYSICS_WEIGHTS}")

Device: cuda
Datasets: data, extrapolation_test.csv
Epochs: 300
Batch size: 256
Collocation points: 8192
Physics weights: [0.0, 0.01, 0.05, 0.1, 0.5, 1.0]


In [3]:
train_df, valid_df, _ = load_data(TRAIN_DATA_PATH, TRAIN_FILE_NAME)
test_df = pd.read_csv(TEST_DATA_PATH / TEST_FILE_NAME)

print(f"Train samples:      {len(train_df):,}")
print(f"Validation samples: {len(valid_df):,}")
print(f"Test samples:       {len(test_df):,}")

train_re = list(map(int, sorted(train_df["re"].unique())))
valid_re = list(map(int, sorted(valid_df["re"].unique())))
test_re = list(map(int, sorted(test_df["re"].unique())))

time_steps = list(map(float, sorted(train_df["time"].unique())))

print(f"\nTraining Reynolds numbers:   {train_re}")
print(f"Validation Reynolds numbers: {valid_re}")
print(f"Test Reynolds numbers:       {test_re}")

print(f"\nTime steps: {time_steps}")

Train samples:      2,364
Validation samples: 473
Test samples:       901,120

Training Reynolds numbers:   [100, 136, 155, 191, 210, 246, 265, 283, 338, 357, 375, 393, 412, 448, 485, 504, 540, 577, 595, 614, 632, 651, 687, 706, 724, 761, 779, 797, 816, 834, 908, 944, 963, 981, 1000]
Validation Reynolds numbers: [173, 302, 467, 522, 742, 871, 926]
Test Reynolds numbers:       [1200, 1400, 1600, 1800, 2000]

Time steps: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]


In [4]:
mean = train_df.mean()
std = train_df.std()

train_df_norm = utils.normalize_data(train_df, mean, std)
valid_df_norm = utils.normalize_data(valid_df, mean, std)

In [5]:
train_dataloader, valid_dataloader, _ = gen_dataloaders(
    train_df_norm,
    valid_df_norm,
    valid_df_norm,
    INPUT_COL_NAMES,
    TARGET_COL_NAMES,
    BATCH_SIZE,
)

In [6]:
ranges = utils.get_domain_ranges(
    train_df,
    INPUT_COL_NAMES,
    overrides={
        "x": (0.0, 1.0),
        "y": (0.0, 1.0),
    },
)

In [ ]:
results = []

for c_physics in PHYSICS_WEIGHTS:
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    print("\n" + "=" * 80)
    print(f"Training model with c_physics = {c_physics}")
    print("=" * 80)

    tag = f"cphys_{c_physics:g}"

    run_dir = utils.create_run_directory(label=f"re_extrapolation_{tag}")

    config = {
        "experiment": "re_extrapolation",
        "physics_weight": c_physics,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "collocation_points": N_COLLOCATION,
        "seed": SEED,
        "train_samples": len(train_df),
        "valid_samples": len(valid_df),
        "test_samples": len(test_df),
    }

    with open(run_dir / "config.json", "w") as f:
        json.dump(config, f, indent=4)

    model = PINN(
        len(INPUT_COL_NAMES),
        len(TARGET_COL_NAMES),
    ).to(device)

    criterion = NavierStokesLoss(
        c_physics,
        mean,
        std,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
    )

    history = train_collocation(
        model=model,
        train_dataloader=train_dataloader,
        valid_dataloader=valid_dataloader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        epochs=EPOCHS,
        run_dir=run_dir,
        input_col_names=INPUT_COL_NAMES,
        ranges=ranges,
        mean=mean,
        std=std,
        n_collocation=N_COLLOCATION,
        train_re_values=train_re,
        valid_re_values=valid_re,
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / "history.csv", index=False)

    fig, _ = utils.plot_training_history(
        history_df,
        output_path=run_dir / "losses.png",
        title=f"Re extrapolation (c_physics={c_physics})",
    )
    plt.close(fig)

    checkpoint = torch.load(
        run_dir / "best_model.pth",
        map_location=device,
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    metrics = utils.evaluate_model(
        model=model,
        df=test_df,
        input_col_names=INPUT_COL_NAMES,
        target_col_names=TARGET_COL_NAMES,
        mean=mean,
        std=std,
        device=device,
    )

    row = {
        "tag": tag,
        "c_physics": c_physics,
        "n_points": len(test_df),
    }

    row.update(utils.flatten_metrics(metrics))

    results.append(row)

    metrics_df = pd.DataFrame([row])
    metrics_df.to_csv(run_dir / "metrics.csv", index=False)

    print(metrics_df[["R2_all", "MAE_all", "RMSE_all"]])

In [8]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["c_physics"]
).reset_index(drop=True)

results_df.to_csv("results/re_extrapolation_results.csv", index=False)

display(results_df)

,tag,c_physics,n_points,MAE_all,MSE_all,RMSE_all,R2_all,MaxAbsError_all,RelL2_all,MAE_U_x,...,RMSE_U_y,R2_U_y,MaxAbsError_U_y,RelL2_U_y,MAE_p,MSE_p,RMSE_p,R2_p,MaxAbsError_p,RelL2_p
0,cphys_0,0.00,901120,0.019927,0.001990,0.044609,0.804323,0.833419,0.442140,0.029634,...,0.044538,0.823899,0.560631,0.419644,0.005127,0.000123,0.011073,0.795838,0.601215,0.421667
1,cphys_0.01,0.01,901120,0.018651,0.001856,0.043078,0.817526,0.841245,0.426964,0.027766,...,0.042553,0.839244,0.549810,0.400943,0.004552,0.000118,0.010846,0.804124,0.599157,0.413022
2,cphys_0.05,0.05,901120,0.017955,0.001828,0.042758,0.820224,0.872732,0.423795,0.027669,...,0.039123,0.864115,0.532075,0.368626,0.005539,0.000134,0.011565,0.777303,0.608849,0.440392
3,cphys_0.1,0.10,901120,0.016326,0.001671,0.040873,0.835728,0.873345,0.405110,0.024952,...,0.037631,0.874283,0.530601,0.354567,0.005198,0.000131,0.011440,0.782081,0.611219,0.435642
4,cphys_0.5,0.50,901120,0.017296,0.001826,0.042734,0.820425,0.920989,0.423559,0.026934,...,0.035044,0.890973,0.551868,0.330192,0.005022,0.000142,0.011908,0.763875,0.632646,0.453475
5,cphys_1,1.00,901120,0.015954,0.001718,0.041444,0.831107,0.893775,0.410767,0.025300,...,0.035841,0.885955,0.557352,0.337705,0.004617,0.000137,0.011720,0.771278,0.629611,0.446310
